# 06 · 回测与评估

> 本 notebook 是《量化研究入门学习资料》第 X 章的可运行配套。
> 数据源：`data/csv/`。运行前请先执行 `python scripts/generate_data.py` 生成数据。

## 目标
重算净值/回撤/绩效指标（对照 `backtest.json`），并演示「删除退市股」造成的幸存者偏差。

In [ ]:
import pandas as pd, numpy as np, json
factors = pd.read_csv("data/csv/factors.csv", parse_dates=["date"])
FACTORS = ["EP", "SIZE", "MOM60", "REV5", "VOL20", "TURN", "ROE", "GROW"]
w = np.full(8, 1 / 8)

def run_portfolio(factors):
    nav, bench, turnover, prev = [1.0], [1.0], [], None
    for _, g in factors.groupby("date"):
        z = g[[f"z_{f}" for f in FACTORS]].fillna(0.0).values @ w
        y = g["next_return"].values
        if np.isnan(y).all():
            nav.append(nav[-1]); bench.append(bench[-1]); turnover.append(0.0); continue
        order = np.argsort(-z); order = order[y[order] == y[order]]
        top = order[:20]
        nav.append(nav[-1] * (1 + y[top].mean())); bench.append(bench[-1] * (1 + np.nanmean(y)))
        to = len(set(top) - (prev or set())) * 2 / 20
        turnover.append(to if prev is not None else 0.0); prev = set(top)
    return nav, bench

nav, bench = run_portfolio(factors)

def metrics(nav, bench):
    r = pd.Series(nav).pct_change().dropna()
    T = len(r)
    ar = (nav[-1] / nav[0]) ** (12 / T) - 1
    av = r.std() * np.sqrt(12)
    dd = (pd.Series(nav) / pd.Series(nav).cummax() - 1).min()
    br = pd.Series(bench).pct_change().dropna()
    er = r - br
    te = er.std() * np.sqrt(12)
    return dict(annual_return=round(ar, 4), sharpe=round(r.mean() * 12 / av, 3) if av else None,
                max_drawdown=round(dd, 4),
                info_ratio=round(er.mean() * 12 / te, 3) if te else None)

m = metrics(nav[1:], bench[1:])          # 去掉初始值，与生成器 60 点口径一致
print("组合:", {k: v for k, v in m.items()})
print("基准年化:", round((bench[-1]) ** (12 / len(bench)) - 1, 4))

In [ ]:
# ---- 对照 backtest.json ----
ref = json.load(open("data/backtest.json", encoding="utf-8"))
ok_nav = np.allclose(nav[1:], ref["nav"]["portfolio"], atol=1e-5)
ok_m = abs(m["annual_return"] - ref["metrics"]["portfolio"]["annual_return"]) < 5e-4
print("净值对照:", "PASS" if ok_nav else "FAIL")
print("年化收益对照:", "PASS" if ok_m else "FAIL")
assert ok_nav and ok_m

### 演示：幸存者偏差
把退市股（000180/000185/000190）从样本中删掉再分层——看分层收益如何被高估：

In [ ]:
# 删除退市股 vs 保留（分层多空收益对比）
basic = pd.read_csv("data/csv/stocks_basic.csv")
delisted = set(basic.dropna(subset=["delist_date"])["code"])
f_no_surv = factors[~factors["code"].isin(delisted)].copy()

def layer_ls(factors, zcol="z_EP"):
    ls = [1.0]
    for _, g in factors.groupby("date"):
        x, y = g[zcol], g["next_return"]
        m = x.notna() & y.notna()
        if m.sum() < 20: ls.append(ls[-1]); continue
        q = pd.qcut(x[m], 10, labels=False)
        ls.append(ls[-1] * (1 + y[m][q == 9].mean() - y[m][q == 0].mean()))
    return np.array(ls)

ls_full = layer_ls(factors)         # 含退市股（正确口径）
ls_surv = layer_ls(f_no_surv)       # 删除退市股（错误口径）
print(f"EP 多空累计收益：正确口径 {ls_full[-1]:.3f}  vs  删除退市股 {ls_surv[-1]:.3f}")
print(f"幸存者偏差高估：{(ls_surv[-1] / ls_full[-1] - 1):.1%}")